In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
df = pd.read_csv("energy consumption.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True)
df = df.sort_values("Timestamp").ffill()
features = ["Electricity_Consumed", "Temperature", "Humidity", "Wind_Speed", "Avg_Past_Consumption"]
a = int(len(df) * 0.7)
b = int(len(df) * 0.85)
train, val, test = df[:a], df[a:b], df[b:]
scaler = StandardScaler()
scaler.fit(train[train["Anomaly_Label"] == 0][features])
window = 24

In [ ]:
# Build train, validation and test windows in this order.
datasets, targets = [], []
for split, frame in enumerate([train, val, test]):
    windows, labels = [], []
    values = scaler.transform(frame[features]).astype("float32")
    faults = frame["Anomaly_Label"].to_numpy()
    for i in range(len(values) - window + 1):
        # Train and validation use only normal windows.
        if split < 2 and faults[i:i + window].max() != 0:
            continue
        windows.append(values[i:i + window])
        labels.append(faults[i + window - 1])
    datasets.append(np.array(windows))
    targets.append(np.array(labels))

x_train, x_val, x_test = datasets
y_test = targets[2]

In [ ]:
model = Sequential([Input((24, 5)),
    LSTM(32),
    RepeatVector(24),
    LSTM(32, return_sequences=True),
    Dense(5)])
model.compile(optimizer="adam", loss="mse")

In [ ]:
model.fit(x_train, x_train, epochs=30, batch_size=128, validation_data=(x_val, x_val))

In [ ]:
normal_error = np.mean((x_val[:, -1] - model.predict(x_val, verbose=0)[:, -1]) ** 2, axis=1)
threshold = np.percentile(normal_error, 95)

In [ ]:
test_error = np.mean((x_test[:, -1] - model.predict(x_test, verbose=0)[:, -1]) ** 2, axis=1)
anomalies = test_error > threshold
print("Test accuracy (%):", round(100 * accuracy_score(y_test, anomalies), 2))
print("Detection AUC:", roc_auc_score(y_test, test_error))